In [35]:
# Import sqlite3 module to work with SQLite databases
import sqlite3

# Import pprint for better formatted printing (optional but useful)
from pprint import pprint

# Create a connection to an in-memory SQLite database
# ":memory:" means the database is temporary and stored in RAM
# It will reset every time the notebook restarts
conn = sqlite3.connect(":memory:")

# This allows us to access columns by name instead of index
# Example: row["name"] instead of row[0]
conn.row_factory = sqlite3.Row

# Create a cursor object
# The cursor is used to execute SQL commands
cur = conn.cursor()


# Define a helper function to run SQL queries
def run_sql(sql, params=None, show=True):
    """
    Executes SQL statements and optionally displays results.

    Parameters:
    sql (str): SQL query to execute
    params (tuple): Optional parameters for parameterized queries
    show (bool): Whether to print results

    Returns:
    rows (list): Query results if available, otherwise None
    """

    # If no parameters provided, use empty tuple
    if params is None:
        params = ()

    # Execute the SQL command
    cur.execute(sql, params)

    # Save changes to database (needed for INSERT, UPDATE, DELETE, CREATE)
    conn.commit()

    # If query returns data (example: SELECT)
    if cur.description:

        # Fetch all rows returned by query
        rows = cur.fetchall()

        # Print results if show=True
        if show:
            print(f"Rows returned: {len(rows)}")

            # Print each row as a dictionary for readability
            for r in rows[:50]:   # limit to first 50 rows
                print(dict(r))

        return rows

    # If no rows returned (example: CREATE, INSERT, DELETE)
    return None


# Confirmation message
print("SQLite database is ready.")

SQLite database is ready.


In [36]:
run_sql("""
CREATE TABLE StudentTBL(
student_id INT PRIMARY KEY,
first_name TEXT NOT NULL,
last_name TEXT NOT NULl
);
""", show=False)

In [37]:
run_sql("""
CREATE TABLE Teachers(
teacher_id INT PRIMARY KEY,
first_name TEXT NOT NULL,
last_name TEXT NOT NULL,
subject TEXT
);
""", show=False)

In [38]:
run_sql("""
CREATE TABLE Classes(
class_id INT PRIMARY KEY,
class_name TEXT NOT NULL,
room_number INT,
teacher_id INT NOT NULL
);
""", show=False)

In [39]:
run_sql("""
INSERT INTO StudentTBL(student_id, first_name, last_name) VALUES
(1, 'Taylor', 'Pendred'),
(2, 'John', 'Doe'),
(3, 'Michael', 'Pendred');
""", show=False)

In [41]:
run_sql("""
INSERT INTO Teachers(teacher_id, first_name, last_name, subject) VALUES
(1, 'Abel', 'Jones', 'Math'),
(2, 'Josh', 'Peters', 'Science'),
(3, 'Frank', 'Castle', 'PE');
""", show=False)

In [45]:
run_sql("""
INSERT INTO Classes(class_id, class_name, room_number, teacher_id) VALUES
(1, 'Math', 5, 1),
(2, 'Science', 3, 2),
(3, 'PE', 7, 3);
""", show=False)

In [47]:
run_sql("SELECT * FROM Classes")

Rows returned: 3
{'class_id': 1, 'class_name': 'Math', 'room_number': 5, 'teacher_id': 1}
{'class_id': 2, 'class_name': 'Science', 'room_number': 3, 'teacher_id': 2}
{'class_id': 3, 'class_name': 'PE', 'room_number': 7, 'teacher_id': 3}


In [48]:
run_sql("SELECT * FROM Teachers")

Rows returned: 3
{'teacher_id': 1, 'first_name': 'Abel', 'last_name': 'Jones', 'subject': 'Math'}
{'teacher_id': 2, 'first_name': 'Josh', 'last_name': 'Peters', 'subject': 'Science'}
{'teacher_id': 3, 'first_name': 'Frank', 'last_name': 'Castle', 'subject': 'PE'}


In [49]:
run_sql("SELECT * FROM StudentTBL")

Rows returned: 3
{'student_id': 1, 'first_name': 'Taylor', 'last_name': 'Pendred'}
{'student_id': 2, 'first_name': 'John', 'last_name': 'Doe'}
{'student_id': 3, 'first_name': 'Michael', 'last_name': 'Pendred'}


In [52]:
run_sql("SELECT class_name FROM Classes WHERE teacher_id=1 OR room_number=3")

Rows returned: 2
{'class_name': 'Math'}
{'class_name': 'Science'}


[<sqlite3.Row at 0x16b722dd1e0>, <sqlite3.Row at 0x16b6f3f4eb0>]

In [53]:
run_sql("SELECT first_name, last_name FROM StudentTBL WHERE student_id IN (1, 2)")

Rows returned: 2
{'first_name': 'Taylor', 'last_name': 'Pendred'}
{'first_name': 'John', 'last_name': 'Doe'}


[<sqlite3.Row at 0x16b7236f730>, <sqlite3.Row at 0x16b722a2d10>]

In [58]:
run_sql("UPDATE Classes SET room_number = null WHERE room_number = 5")

In [68]:
run_sql("SELECT * FROM Classes WHERE room_number IS NULL")

Rows returned: 1
{'class_id': 1, 'class_name': 'Math', 'room_number': None, 'teacher_id': 1}


In [71]:
run_sql("SELECT * FROM StudentTBL ORDER BY last_name ASC LIMIT 2")

Rows returned: 2
{'student_id': 2, 'first_name': 'John', 'last_name': 'Doe'}
{'student_id': 1, 'first_name': 'Taylor', 'last_name': 'Pendred'}


[<sqlite3.Row at 0x16b72360790>, <sqlite3.Row at 0x16b723610c0>]

In [81]:
run_sql("SELECT COUNT(class_id) as Total_Classes_Taught FROM Classes WHERE teacher_id = 2")

Rows returned: 1
{'Total_Classes_Taught': 1}


In [84]:
run_sql("SELECT teacher_id, COUNT(class_id) as Total_Classes_Taught FROM Classes GROUP BY teacher_id ORDER BY Total_Classes_Taught ASC")

Rows returned: 3
{'teacher_id': 1, 'Total_Classes_Taught': 1}
{'teacher_id': 2, 'Total_Classes_Taught': 1}
{'teacher_id': 3, 'Total_Classes_Taught': 1}


In [89]:
run_sql("SELECT DISTINCT COUNT(room_number) as Total_Rooms FROM Classes")
print("We removed 1 of the 3 rooms by setting the value of the room to null in the previous tasks")

Rows returned: 1
{'Total_Rooms': 2}
We removed 1 of the 3 rooms by setting the value of the room to null in the previous tasks


In [94]:
run_sql("ALTER TABLE Classes ADD Semester INT")

In [95]:
run_sql("SELECT * FROM Classes")

Rows returned: 3
{'class_id': 1, 'class_name': 'Math', 'room_number': None, 'teacher_id': 1, 'Semester': None}
{'class_id': 2, 'class_name': 'Science', 'room_number': 3, 'teacher_id': 2, 'Semester': None}
{'class_id': 3, 'class_name': 'PE', 'room_number': 7, 'teacher_id': 3, 'Semester': None}


In [99]:
run_sql("UPDATE Classes SET Semester = 2 WHERE Semester IS NULL")


In [100]:
run_sql("SELECT * FROM Classes")

Rows returned: 3
{'class_id': 1, 'class_name': 'Math', 'room_number': None, 'teacher_id': 1, 'Semester': 2}
{'class_id': 2, 'class_name': 'Science', 'room_number': 3, 'teacher_id': 2, 'Semester': 2}
{'class_id': 3, 'class_name': 'PE', 'room_number': 7, 'teacher_id': 3, 'Semester': 2}


In [101]:
run_sql("DELETE FROM Classes WHERE room_number IS NULL")

In [102]:
run_sql("SELECT * FROM Classes")

Rows returned: 2
{'class_id': 2, 'class_name': 'Science', 'room_number': 3, 'teacher_id': 2, 'Semester': 2}
{'class_id': 3, 'class_name': 'PE', 'room_number': 7, 'teacher_id': 3, 'Semester': 2}


[<sqlite3.Row at 0x16b724460b0>, <sqlite3.Row at 0x16b72559f00>]